# Fine-tune a pretrained language model

Change the model and dataset values, then run the cells. TrainLM handles tokenization, training, evaluation, and local export. The three-step default is only a smoke test; it does not measure model quality. The model is not uploaded.

Use a Colab CPU or GPU runtime for this notebook. Native TorchTPU fine-tuning is not validated yet. See the [fine-tuning guide](../docs/tutorials/TEXT_FINETUNING.md) for longer runs, custom data, best-checkpoint selection, and resume.

In [ ]:
import subprocess
import sys

source = "trainlm @ git+https://github.com/Dhiraj309/TrainLM.git@main"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", source],
    check=True,
)
requirements = [
    "huggingface-hub>=1.5,<2",
    "PyYAML>=6,<7",
    "transformers>=5.15,<6",
    "datasets>=3,<5",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements],
    check=True,
)
print("TrainLM installed without asking pip to replace PyTorch or TorchTPU.")

In [ ]:
from trainlm import TrainLMFineTuningConfig, TrainLMTrainingArguments, run_finetuning
MODEL = "sshleifer/tiny-gpt2"
DATASET = "roneneldan/TinyStories"
DATASET_REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64"
# Keep this smoke run quick by limiting data preparation too.
OUTPUT_DIR = "trainlm-finetuning"

config = TrainLMFineTuningConfig(
    model_name_or_path=MODEL,
    dataset_id=DATASET,
    dataset_revision=DATASET_REVISION,
    text_field="text",
    train_split="train[:1000]",
    validation_split="validation[:100]",
    args=TrainLMTrainingArguments(
        output_dir=OUTPUT_DIR,
        max_steps=3,  # Increase this after the smoke run succeeds.
        sequence_length=128,
        # run_finetuning performs one final evaluation automatically.
        eval_strategy="no",
        save_strategy="no",
        accelerator="auto",
    ),
)
result = run_finetuning(config)
print(result.summary())
print(f"model:  {result.model_path}")
print(f"report: {result.summary_path}")

## Download the model

The ZIP contains only the exported model. The run summary remains at the path printed above.

In [ ]:
import shutil

archive = shutil.make_archive(
    "trainlm-finetuned-model",
    "zip",
    root_dir=OUTPUT_DIR,
    base_dir="model",
)
try:
    from google.colab import files
except ImportError:
    print(f"Model archive: {archive}")
else:
    files.download(archive)